# Relabeled Tag Data Explorer

Load all `data_relabeled*.csv` files, and browse by new category.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import sys
sys.path.insert(0, '.')
from tag_labels import TAG_LABELS

files = sorted(glob.glob('data_relabeled*.csv'))
dfs = []
for f in files:
    d = pd.read_csv(f)
    d['source'] = f
    dfs.append(d)
df = pd.concat(dfs, ignore_index=True)
df['domain'] = df['new_label'].map(lambda x: TAG_LABELS.get(x, {}).get('domain', '?'))
df['description'] = df['new_label'].map(lambda x: TAG_LABELS.get(x, {}).get('description', ''))
print(f'Loaded {len(files)} files: {files}')
print(f'Shape: {df.shape}')
df.head(10)

## Source File Distribution

In [ ]:
print('Rows per source file:')
for src, grp in df.groupby('source'):
    print(f'  {src}: {len(grp)} rows, {grp["new_label"].nunique()} unique labels')

## New Label Distribution (tag count)

In [ ]:
label_dist = df['new_label'].value_counts()
fig, ax = plt.subplots(figsize=(10, 8))
label_dist.plot.barh(ax=ax)
ax.set_xlabel('Number of tags')
ax.set_title('New Label Distribution')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## New Label Distribution (frequency = sum of count)

In [ ]:
freq = df.groupby('new_label')['count'].sum().sort_values(ascending=False).to_frame('frequency')
freq['domain'] = freq.index.map(lambda x: TAG_LABELS.get(x, {}).get('domain', '?'))
freq['description'] = freq.index.map(lambda x: TAG_LABELS.get(x, {}).get('description', ''))
freq

## Domain Distribution

In [ ]:
domain_freq = df.groupby('domain')['count'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
domain_freq.plot.bar(ax=ax, color='steelblue')
ax.set_ylabel('Frequency')
ax.set_title('Domain Distribution (by frequency)')
plt.tight_layout()
plt.show()

domain_tags = df['domain'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
domain_tags.plot.bar(ax=ax, color='coral')
ax.set_ylabel('Number of tags')
ax.set_title('Domain Distribution (by tag count)')
plt.tight_layout()
plt.show()

## Filter by New Label

Change the value below to view a different category.

In [ ]:
new_label = 'willing_to_pay'  # <- change this

filtered = df[df['new_label'] == new_label].sort_values('count', ascending=False)
info = TAG_LABELS.get(new_label, {})
print(f'Label: {new_label}')
print(f'Domain: {info.get("domain", "?")}')
print(f'Description: {info.get("description", "")}')
print(f'{len(filtered)} tags  |  frequency: {filtered["count"].sum()}')
filtered

## Filter by Domain

In [ ]:
domain = 'Financial'  # <- change this

filtered = df[df['domain'] == domain].sort_values('count', ascending=False)
labels_in_domain = filtered['new_label'].unique()
print(f'Domain: {domain}')
print(f'Labels: {sorted(labels_in_domain)}')
print(f'{len(filtered)} tags  |  frequency: {filtered["count"].sum()}')
filtered.head(30)

## All New Labels Quick View

In [ ]:
for label in df.groupby('new_label')['count'].sum().sort_values(ascending=False).index:
    subset = df[df['new_label'] == label].sort_values('count', ascending=False)
    info = TAG_LABELS.get(label, {})
    top3 = ', '.join(f'{r.tag}({r["count"]})' for _, r in subset.head(3).iterrows())
    print(f'{label:30s} [{info.get("domain","?"):10s}] freq={subset["count"].sum():>5d} tags={len(subset):>4d}  top3: {top3}')

## Unclassified tags

In [ ]:
unclassified = df[df['new_label'] == 'unclassified']
print(f'{len(unclassified)} unclassified tags')
unclassified